# Experiment 1 — Which surprisal works best? (TF-IDF + one surprisal)

For every surprisal column we train a logistic regression with **TF-IDF text + that one surprisal**,
on all three master sheets:

| File | One row is | Label | Features |
|---|---|---|---|
| `master-absolute.csv` | one paragraph version | `edit` = 1, `base` = 0 | TF-IDF(text) + surprisal |
| `master-pairs-cumulative.csv` | pair `(v_0, v_k)` | is B the edit? | TF-IDF(B) − TF-IDF(A) + surprisal delta |
| `master-pairs-step.csv` | pair `(v_{k-1}, v_k)` | is B the edit? | TF-IDF(B) − TF-IDF(A) + surprisal delta |

We also run **TF-IDF only** as a baseline, so we can see how much each surprisal adds.

The test set is the same 20% of articles used in the other logistic notebooks.
Results are saved to `output/paragraph/tables/exp1-tfidf-surprisal.csv`.

## 1. Settings

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/master/ to read, output/<LEVEL>/tables/ to write
master_dir = Path(f"../../data/{LEVEL}/master")  # master sheets + feature-engineering/
tables_dir = Path(f"../../output/{LEVEL}/tables")  # final result tables
results_path = tables_dir / "exp1-tfidf-surprisal.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2

# All surprisal columns: GPT-2, then 4 corpora x n-gram order 1..5
# (_1 = unigram, _2 = bigram, _3 = trigram, ...)
SURPRISALS = ["mean_surprisal"]
for corpus in ["simplewiki", "ptb_wsj", "wikitext2", "brown"]:
    for n in [1, 2, 3, 4, 5]:
        SURPRISALS.append(f"mean_surprisal_{corpus}_{n}")

print(len(SURPRISALS), "surprisal features")
print(SURPRISALS)

## 2. Train / test split (by article)

In [ ]:
# one shared list of test articles for both levels, so paragraph and sentence results are measured on the same
# articles. Chosen once from the paragraph master sheet with the same SEED / TEST_SIZE as before (so the paragraph
# split is unchanged) and saved; every experiment at every level reads it from then on.
test_articles_file = Path("../../data/test-articles.csv")
if not test_articles_file.exists():
    paragraph_articles = pd.read_csv("../../data/paragraph/master/master-absolute.csv", usecols=["article"])["article"].astype(str)
    unique_paragraph_articles = np.array(sorted(paragraph_articles.unique()))
    n_test = int(round(len(unique_paragraph_articles) * TEST_SIZE))
    chosen = sorted(np.random.default_rng(SEED).permutation(unique_paragraph_articles)[:n_test])
    pd.DataFrame({"article": chosen}).to_csv(test_articles_file, index=False)
test_articles = set(pd.read_csv(test_articles_file)["article"].astype(str))
articles = pd.read_csv(master_dir / "master-absolute.csv", usecols=["article"])["article"].astype(str)
unique_articles = np.array(sorted(articles.unique()))
print("test articles:", len(test_articles & set(unique_articles)), "of", len(unique_articles))

# paragraphs where a metric could not be calculated (one word headings like "Steps") - made by feature-engineering.ipynb
# every pair that has one of these paragraphs (as base or as edit) is left out
undefined = pd.read_csv(master_dir / "feature-engineering" / "undefined_paragraphs.csv")
undefined_keys = set(zip(undefined["article"].astype(str), undefined["revision_group_no"], undefined["revision_depth"]))
print("undefined paragraphs:", len(undefined_keys))

def defined_pairs(df, base_depth_column, edit_depth_column):
    #True for every pair where both paragraphs have all metrics
    base_keys = zip(df["article"].astype(str), df["revision_group_no"], df[base_depth_column])
    edit_keys = zip(df["article"].astype(str), df["revision_group_no"], df[edit_depth_column])
    return np.array([base not in undefined_keys and edit not in undefined_keys for base, edit in zip(base_keys, edit_keys)])

## 3. Two small helpers

- `train_and_test` fits a logistic regression and returns accuracy and AUC.
- `add_surprisal` scales one surprisal column and glues it next to the TF-IDF columns.

In [ ]:
def train_and_test(X_train, y_train, X_test, y_test):
    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, predicted)
    auc = roc_auc_score(y_test, probability)
    return accuracy, auc


def add_surprisal(tfidf_train, tfidf_test, values_train, values_test):
    # scale so surprisal has mean 0 and std 1 (learned on train only)
    scaler = StandardScaler()
    values_train = scaler.fit_transform(values_train.reshape(-1, 1))
    values_test = scaler.transform(values_test.reshape(-1, 1))
    # put the surprisal column next to the TF-IDF columns
    X_train = hstack([tfidf_train, csr_matrix(values_train)]).tocsr()
    X_test = hstack([tfidf_test, csr_matrix(values_test)]).tocsr()
    return X_train, X_test


results = []  # one dict per (file, surprisal)

## 4. Absolute file

Each row is one paragraph. The model learns: does this text look like an `edit` or a `base`?

In [ ]:
df = pd.read_csv(
    master_dir / "master-absolute.csv",
    usecols=["article", "revision_group_no", "pair_older_depth", "pair_newer_depth", "label", "text", *SURPRISALS],
    low_memory=False,
)
keep = defined_pairs(df, "pair_older_depth", "pair_newer_depth")
print("left out (pair has an undefined paragraph):", (~keep).sum(), "rows")
df = df[keep].reset_index(drop=True)
text = df["text"].fillna("").astype(str)
y = (df["label"] == "edit").astype(int).to_numpy()
is_test = df["article"].astype(str).isin(test_articles).to_numpy()
is_train = ~is_test
print("train rows:", is_train.sum(), " test rows:", is_test.sum())

# TF-IDF is built once and reused for every surprisal
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
tfidf_train = vectorizer.fit_transform(text[is_train])
tfidf_test = vectorizer.transform(text[is_test])

# Baseline: TF-IDF only
acc, auc = train_and_test(tfidf_train, y[is_train], tfidf_test, y[is_test])
results.append({"file": "absolute", "surprisal": "tfidf_only", "accuracy": acc, "auc": auc})
print(f"tfidf_only                    acc={acc:.4f}  auc={auc:.4f}")

# TF-IDF + one surprisal at a time
for name in SURPRISALS:
    values = df[name].fillna(0).to_numpy(dtype=float)
    X_train, X_test = add_surprisal(tfidf_train, tfidf_test, values[is_train], values[is_test])
    acc, auc = train_and_test(X_train, y[is_train], X_test, y[is_test])
    results.append({"file": "absolute", "surprisal": name, "accuracy": acc, "auc": auc})
    print(f"{name:30s} acc={acc:.4f}  auc={auc:.4f}")

## 5. Pair files (cumulative and step)

Each row is a pair (base, edit). We randomly show it as (A, B) or (B, A) so the model can't
just learn "B is always the edit". The label says whether **B is the edit**.

- text feature: `TF-IDF(B) − TF-IDF(A)`
- surprisal feature: `surprisal(B) − surprisal(A)` (this is `delta_*` from the sheet, flipped when we swapped)

In [ ]:
for setting in ["cumulative", "step"]:
    delta_cols = ["delta_" + name for name in SURPRISALS]
    df = pd.read_csv(
        master_dir / f"master-pairs-{setting}.csv",
        usecols=["article", "revision_group_no", "base_depth", "edit_depth", "base_text", "edit_text", *delta_cols],
        low_memory=False,
    )
    keep = defined_pairs(df, "base_depth", "edit_depth")
    print("left out (pair has an undefined paragraph):", (~keep).sum(), "pairs")
    df = df[keep].reset_index(drop=True)
    base_text = df["base_text"].fillna("").astype(str).to_numpy(dtype=object)
    edit_text = df["edit_text"].fillna("").astype(str).to_numpy(dtype=object)

    # swap = True  ->  A is the edit, B is the base, label 0
    # swap = False ->  A is the base, B is the edit, label 1
    swap = np.random.default_rng(SEED).integers(0, 2, size=len(df)).astype(bool)
    A = np.where(swap, edit_text, base_text)
    B = np.where(swap, base_text, edit_text)
    y = (~swap).astype(int)
    sign = np.where(swap, -1.0, 1.0)  # delta is (edit - base); flip it when swapped

    is_test = df["article"].astype(str).isin(test_articles).to_numpy()
    is_train = ~is_test
    print(f"\n=== {setting} ===  train pairs: {is_train.sum()}  test pairs: {is_test.sum()}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    vectorizer.fit(list(A[is_train]) + list(B[is_train]))
    tfidf_train = vectorizer.transform(B[is_train]) - vectorizer.transform(A[is_train])
    tfidf_test = vectorizer.transform(B[is_test]) - vectorizer.transform(A[is_test])

    # Baseline: TF-IDF only
    acc, auc = train_and_test(tfidf_train, y[is_train], tfidf_test, y[is_test])
    results.append({"file": setting, "surprisal": "tfidf_only", "accuracy": acc, "auc": auc})
    print(f"tfidf_only                    acc={acc:.4f}  auc={auc:.4f}")

    # TF-IDF + one surprisal delta at a time
    for name in SURPRISALS:
        values = df["delta_" + name].fillna(0).to_numpy(dtype=float) * sign
        X_train, X_test = add_surprisal(tfidf_train, tfidf_test, values[is_train], values[is_test])
        acc, auc = train_and_test(X_train, y[is_train], X_test, y[is_test])
        results.append({"file": setting, "surprisal": name, "accuracy": acc, "auc": auc})
        print(f"{name:30s} acc={acc:.4f}  auc={auc:.4f}")

## 6. Results table

In [ ]:
results_df = pd.DataFrame(results)
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print("saved", results_path)

# Accuracy: one row per surprisal, one column per file
accuracy_table = results_df.pivot(index="surprisal", columns="file", values="accuracy")
accuracy_table = accuracy_table[["absolute", "cumulative", "step"]]
accuracy_table["mean"] = accuracy_table.mean(axis=1)
accuracy_table = accuracy_table.sort_values("mean", ascending=False)
accuracy_table.round(4)

In [ ]:
# Same table for AUC
auc_table = results_df.pivot(index="surprisal", columns="file", values="auc")
auc_table = auc_table[["absolute", "cumulative", "step"]]
auc_table["mean"] = auc_table.mean(axis=1)
auc_table = auc_table.sort_values("mean", ascending=False)
auc_table.round(4)

In [ ]:
# How much does each surprisal add on top of TF-IDF only? (accuracy points)
gain_table = accuracy_table - accuracy_table.loc["tfidf_only"]
gain_table = gain_table.drop(index="tfidf_only")
(gain_table * 100).round(2)